In [3]:
import pandas as pd
import yaml
from argparse import ArgumentParser
from dotenv import load_dotenv
from typing import Literal
from util.llm import *
from util.prompt_loader import PromptLoader

load_dotenv()

False

In [4]:
from util.logging import logging

log = logging.info
err = logging.error

# Variáveis de Configuração

In [5]:
MODEL    = "modelo"
PROVIDER = "exemplo"  # "ollama" ou "local-openai" ou "huggingface"
# número de amostras para self-consistency, caso não queira, deixe 1
NUM_SAMPLES = 3

CURRENT_BENCHMARK           = "nome do benchmark"
TABLE_DESCRIPTIONS_PATH    = Path(CURRENT_BENCHMARK) / "table_descriptions.csv"
COLUMN_DESCRIPTIONS_PATH   = Path(CURRENT_BENCHMARK) / "column_descriptions_with_cta.csv"
TABLE_CARTESIANS_PATH       = Path(CURRENT_BENCHMARK) / "table_cartesians.csv"
GROUND_TRUTH_PATH           = Path(CURRENT_BENCHMARK) / "groundTruth.csv"

# Porta para se comunicar com a API; 11434 é o padrão do Ollama
PORT                 = 11434

# Outros caminhos (não precisa mexer)
THINK                = False
PREDICTIONS_DIR_PATH = Path(f"data/predictions/{CURRENT_BENCHMARK}/{MODEL}") if not THINK else Path(f"data/predictions/{CURRENT_BENCHMARK}/{MODEL}_think")


---

In [ ]:
table_descriptions  = pd.read_csv(TABLE_DESCRIPTIONS_PATH)
column_descriptions = pd.read_csv(COLUMN_DESCRIPTIONS_PATH, sep=';')
table_cartesians    = pd.read_csv(TABLE_CARTESIANS_PATH)

In [ ]:
prompt_loader = PromptLoader()

In [ ]:
if PROVIDER == "ollama":
    service = OllamaService({
        "model":    MODEL,
        "base_url": f"http://localhost:{PORT}",
        "host": "localhost",
        "think": THINK
    })
elif PROVIDER == "vllm" or PROVIDER == "local-openai":
    service = LocalOpenAILikeService(
        model=MODEL,
        port=PORT,
    )
elif PROVIDER == "huggingface":
    service = HuggingFaceService({
        # "api_key": "hf_key",
        "model": MODEL
    })
else:
    raise ValueError(f"Unsupported provider: {PROVIDER}")

In [ ]:
print(f"Using model: {MODEL} on port {PORT} with provider {PROVIDER} (think={THINK})")

In [ ]:
def build_prompt_for_cartesian_idx(i: int) -> tuple[str, str, str, str]:
    """Build prompt for a single cartesian pair row index, with *no* preprocessing.

    :return: (left_table, right_table, system_msg, user_msg)
    """
    left_table = table_cartesians.iloc[i, 0]
    right_table = table_cartesians.iloc[i, 1]

    left_td = table_descriptions[table_descriptions['TableName'] == left_table]
    right_td = table_descriptions[table_descriptions['TableName'] == right_table]

    if left_td.empty or right_td.empty:
        raise ValueError(f"Missing table description (left={left_td.empty}, right={right_td.empty})")

    left_table_description = left_td.iloc[0, 1]
    right_table_description = right_td.iloc[0, 1]

    left_cd = column_descriptions[column_descriptions['TableName'] == left_table]
    right_cd = column_descriptions[column_descriptions['TableName'] == right_table]

    if left_cd.empty or right_cd.empty:
        raise ValueError(f"Missing column descriptions (left={left_cd.empty}, right={right_cd.empty})")

    left_column_names = left_cd.iloc[:, 1].values
    left_column_descriptions = left_cd.iloc[:, 2].values

    right_column_names = right_cd.iloc[:, 1].values
    right_column_descriptions = right_cd.iloc[:, 2].values

    system_msg = prompt_loader.system_prompt()
    user_msg = prompt_loader.user_prompt(
        query_table_name=left_table,
        query_table_description=left_table_description,
        query_table_column_names=left_column_names,
        query_table_column_descriptions=left_column_descriptions,
        candidate_table_name=right_table,
        candidate_table_description=right_table_description,
        candidate_table_colunm_names=right_column_names,
        candidate_table_column_descriptions=right_column_descriptions,
    )

    return left_table, right_table, system_msg, user_msg

# Helpers de I/O incremental

def load_existing_structured(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def append_structured_row(path: Path, row: dict):
    df_row = pd.DataFrame([row])
    if not path.exists():
        df_row.to_csv(path, index=False)
    else:
        df_row.to_csv(path, mode="a", header=False, index=False)


# Modelo de resposta estruturada

class JoinDiscoveryResponse(BaseModel):
    table_joinable:  Literal["Yes", "No"]
    column_joinable: Literal["Yes", "No"]
    target_match:    list[str] | str | None
    candidate_match: list[str] | str | None


In [ ]:

def run_structured_incremental(
    llm: LLMService,
    output_path: Path,
):
    """Itera sobre `table_cartesians` (por índice), retomando do ponto onde parou.

    We intentionally don't precompute prompts: each idx builds its prompt lazily.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    existing_df = load_existing_structured(output_path)
    start_idx   = len(existing_df)

    total = len(table_cartesians)
    log(f"Resuming from {start_idx} / {total}...")

    for i in range(start_idx, total):
        try:
            left_t, right_t, system_msg, user_msg = build_prompt_for_cartesian_idx(i)
        except Exception as e:
            err(f"Prompt build failed at idx {i}: {e}")
            out_row = {
                "idx":             i,
                "target_table":    table_cartesians.iloc[i, 0],
                "candidate_table": table_cartesians.iloc[i, 1],
                "table_joinable":  None,
                "column_joinable": None,
                "target_match":    None,
                "candidate_match": None,
                "error":           f"prompt_build_error: {e}",
            }
            append_structured_row(output_path, out_row)
            continue

        log(f"Processing idx {i} — Target: {left_t} | Candidate: {right_t}")
        log(f"System Message length: {len(system_msg)} | User Message length: {len(user_msg)}")

        try:
            response = llm.generate_structured(
                system_msg=system_msg,
                user_msg=user_msg,
                base_model=JoinDiscoveryResponse,
            )

            out_row = {
                "idx":             i,
                "target_table":    left_t,
                "candidate_table": right_t,
                "table_joinable":  response.table_joinable,
                "column_joinable": response.column_joinable,
                "target_match":    response.target_match,
                "candidate_match": response.candidate_match,
            }

        except Exception as e:
            err(f"Validation failed at idx {i}: {e}")

            out_row = {
                "idx":             i,
                "target_table":    left_t,
                "candidate_table": right_t,
                "table_joinable":  None,
                "column_joinable": None,
                "target_match":    None,
                "candidate_match": None,
                "error":           str(e),
            }

        append_structured_row(output_path, out_row)
        log(f"Saved {i + 1}/{total}")

In [ ]:
def run_self_consistency(
    llm_service: LLMService,
    output_dir: Path,
    self_consistency_samples: int,
):
    """Executa `self_consistency_samples` rodadas de inferência,
    salvando cada uma em predictions_1.csv, predictions_2.csv, …
    """
    for i in range(1, self_consistency_samples + 1):
        print(f"Running self-consistency iteration {i}/{self_consistency_samples}...")
        output_path = output_dir / f"predictions_{i}.csv"
        run_structured_incremental(llm_service, output_path)

# Rodar o Programa

In [ ]:
run_self_consistency(
    service,
    PREDICTIONS_DIR_PATH,
    NUM_SAMPLES,
)
